# Machine Learning Notes
## Day 25: Feature Scaling — Normalization (MinMaxScaler) — Answer Key

> **Watermark:** Amol Jagtap | amoljagtap3001@gmail.com  
> **Topic:** Min-Max Scaling / Normalization  
> **Difficulty:** Beginner to Intermediate  

---
### Note:
This notebook contains FULLY WORKED SOLUTIONS for every exercise in 
**Day25_Normalization_Practice_Questions.ipynb**. Use this to check your work 
or to study the reference implementation.

### Topics Covered:
1. Manual Min-Max scaling calculation
2. Using MinMaxScaler from scikit-learn
3. Custom feature_range scaling
4. Correct train-test split workflow (no data leakage)
5. Visualizing the effect of normalization
6. Outlier sensitivity demonstration
7. Normalization vs Standardization comparison
8. Mini end-to-end pipeline using MinMaxScaler

---

In [ ]:
# ============================================================
# SETUP — Run this first!
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsClassifier

# Pretty plots
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.titlesize'] = 14
sns.set_style('whitegrid')

print('All libraries imported successfully!')
print('Notebook by: Amol Jagtap | amoljagtap3001@gmail.com')

---
## Section 1: Manual Min-Max Scaling Calculation

**Concept Recap:**
- Formula: `x' = (x - x_min) / (x_max - x_min)`
- x_min maps to 0, x_max maps to 1, everything else scales proportionally in between
- Always computed using TRAINING data statistics only

In [ ]:
# ============================================================
# ANSWER 1: Manual Min-Max Scaling
# ============================================================

# Step 1: find min and max
x_min = marks.min()
x_max = marks.max()
print(f'x_min = {x_min}, x_max = {x_max}, range = {x_max - x_min}')

# Step 2: apply formula (vectorized — works on the whole series at once)
scaled_marks = (marks - x_min) / (x_max - x_min)

# Step 3: verify bounds
print(f'\nMin of scaled output: {scaled_marks.min():.4f}  (should be 0.0000)')
print(f'Max of scaled output: {scaled_marks.max():.4f}  (should be 1.0000)')

# Step 4: comparison table
result = pd.DataFrame({
    'Original_Marks': marks,
    'Scaled_Marks':   scaled_marks.round(4)
})
print('\nComparison Table:')
print(result.to_string(index=False))

---
## Section 2: MinMaxScaler from scikit-learn

**Concept Recap:**
- `scaler.fit(X_train)` learns x_min and x_max from training data
- `scaler.transform(X)` applies the formula
- `scaler.fit_transform(X_train)` does both in one call (training data only!)
- Attributes: `scaler.data_min_`, `scaler.data_max_`, `scaler.data_range_`

In [ ]:
# ============================================================
# ANSWER 2: MinMaxScaler Usage
# ============================================================

scaler = MinMaxScaler(feature_range=(0, 1))

scaled_array = scaler.fit_transform(df)
scaled_df = pd.DataFrame(scaled_array, columns=df.columns).round(4)

print('Scaled Data:')
print(scaled_df)

print('\nLearned statistics from training data:')
print('data_min_  :', scaler.data_min_)
print('data_max_  :', scaler.data_max_)
print('data_range_:', scaler.data_range_)

print('\nSanity check — every column min should be 0, max should be 1:')
print(scaled_df.describe().loc[['min', 'max']])

---
## Section 3: Custom feature_range Scaling

**Concept Recap:**
- Default range is [0, 1]
- General formula: `x' = a + [(x - x_min)/(x_max - x_min)] * (b - a)`
- Common alternative ranges: [-1, 1] for tanh-based neural nets, [0, 100] for percentage-style output

In [ ]:
# ============================================================
# ANSWER 3: Custom Range Scaling
# ============================================================

scaler_neg1to1 = MinMaxScaler(feature_range=(-1, 1))
scaler_0to100  = MinMaxScaler(feature_range=(0, 100))

scaled_neg1to1 = scaler_neg1to1.fit_transform(temperature)
scaled_0to100  = scaler_0to100.fit_transform(temperature)

comparison = pd.DataFrame({
    'Original_Celsius': temperature['Temp_C'],
    'Scaled_[-1,1]':    scaled_neg1to1.flatten().round(3),
    'Scaled_[0,100]':   scaled_0to100.flatten().round(2)
})
print(comparison.to_string(index=False))

print(f'\n[-1,1] range  -> min: {scaled_neg1to1.min()}, max: {scaled_neg1to1.max()}')
print(f'[0,100] range -> min: {scaled_0to100.min()}, max: {scaled_0to100.max()}')

print('\nNote: The coldest temperature (-10) always maps to the lower bound')
print('      and the hottest temperature (45) always maps to the upper bound.')

---
## Section 4: Correct Train-Test Split Workflow (No Data Leakage)

**Concept Recap:**
- ALWAYS split data first, THEN fit the scaler on training data only
- Test data is transformed using statistics learned from training data
- Fitting on the full dataset before splitting = data leakage = inflated test performance

In [ ]:
# ============================================================
# ANSWER 4: Data Leakage Demonstration
# ============================================================

# ----- WRONG WAY -----
# Fitting on the full dataset before splitting means the scaler has
# already "seen" the test set's min/max values — this is data leakage.
wrong_scaler = MinMaxScaler()
X_scaled_wrong = wrong_scaler.fit_transform(X)          # fit on FULL data
X_train_w, X_test_w = train_test_split(X_scaled_wrong, test_size=0.2, random_state=42)

print('WRONG WAY — scaler fitted on full dataset:')
print('  data_min_:', wrong_scaler.data_min_.round(2))
print('  data_max_:', wrong_scaler.data_max_.round(2))

# ----- CORRECT WAY -----
# Split first, then fit scaler ONLY on training data
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

correct_scaler = MinMaxScaler()
correct_scaler.fit(X_train)                              # fit on TRAIN only
X_train_sc = correct_scaler.transform(X_train)
X_test_sc  = correct_scaler.transform(X_test)

print('\nCORRECT WAY — scaler fitted on training data only:')
print('  data_min_:', correct_scaler.data_min_.round(2))
print('  data_max_:', correct_scaler.data_max_.round(2))

print('\nObservation: The learned min/max differ between the two approaches.')
print('In the WRONG way, test-set extreme values influenced the scaling')
print('used for training data — the model indirectly "saw" the test set.')
print('This makes evaluation metrics on the test set overly optimistic.')

# Check if any test value in the correct workflow falls outside [0,1]
# (this can legitimately happen since test min/max were not used to fit)
print(f'\nCorrect workflow — test data scaled range: '
      f'[{X_test_sc.min():.3f}, {X_test_sc.max():.3f}]')
print('(Values outside [0,1] are expected and fine — they reflect real generalization.)')

---
## Section 5: Visualizing the Effect of Normalization

**Concept Recap:**
- Normalization changes the numeric range but preserves the SHAPE of the distribution
- Plotting before/after side by side makes this clear

In [ ]:
# ============================================================
# ANSWER 5: Visualizing Normalization Effect
# ============================================================

income_df = pd.DataFrame({'Income': income})
scaler = MinMaxScaler()
income_scaled = scaler.fit_transform(income_df).flatten()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(income, bins=50, color='#1565C0', edgecolor='white', alpha=0.85)
axes[0].set_title(f'Original Income\nSkewness = {pd.Series(income).skew():.3f}', fontsize=13)
axes[0].set_xlabel('Income (₹)')
axes[0].set_ylabel('Frequency')

axes[1].hist(income_scaled, bins=50, color='#2E7D32', edgecolor='white', alpha=0.85)
axes[1].set_title(f'Min-Max Scaled Income [0,1]\nSkewness = {pd.Series(income_scaled).skew():.3f}', fontsize=13)
axes[1].set_xlabel('Scaled Income')
axes[1].set_ylabel('Frequency')

plt.suptitle('Effect of Min-Max Scaling on Distribution Shape\n'
             'Amol Jagtap | amoljagtap3001@gmail.com', fontsize=11, color='gray', y=1.02)
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/Day25_normalization_shape.png', dpi=120, bbox_inches='tight')
plt.show()

print(f'Original skewness : {pd.Series(income).skew():.4f}')
print(f'Scaled skewness   : {pd.Series(income_scaled).skew():.4f}')
print('\nConclusion: Skewness is identical (within rounding) — the SHAPE of')
print('the distribution is unchanged. Only the numeric RANGE changed to [0,1].')

---
## Section 6: Outlier Sensitivity Demonstration

**Concept Recap:**
- MinMaxScaler uses x_min and x_max — the most extreme values
- One outlier can crush all normal values into a tiny sub-range near 0 or 1
- This is the biggest weakness of normalization compared to standardization

In [ ]:
# ============================================================
# ANSWER 6: Outlier Sensitivity
# ============================================================

scaler_clean   = MinMaxScaler()
scaler_outlier = MinMaxScaler()

scaled_clean   = scaler_clean.fit_transform(salaries_clean).flatten()
scaled_outlier = scaler_outlier.fit_transform(salaries_outlier).flatten()

print('WITHOUT outlier — scaled salaries:')
for orig, sc in zip(salaries_clean['Salary'], scaled_clean):
    print(f'  {orig:>8,} -> {sc:.4f}')

print('\nWITH outlier (1,000,000 added) — scaled salaries:')
for orig, sc in zip(salaries_outlier['Salary'], scaled_outlier):
    print(f'  {orig:>8,} -> {sc:.4f}')

# Range occupied by the 6 "normal" salaries (excluding the outlier)
normal_range_clean   = scaled_clean.max() - scaled_clean.min()
normal_range_outlier = scaled_outlier[:-1].max() - scaled_outlier[:-1].min()

print(f'\nRange occupied by normal salaries WITHOUT outlier: {normal_range_clean:.4f}  (out of 1.0)')
print(f'Range occupied by normal salaries WITH outlier:    {normal_range_outlier:.4f}  (out of 1.0)')

# Visual comparison
fig, axes = plt.subplots(2, 1, figsize=(10, 4))
axes[0].scatter(scaled_clean, [0]*len(scaled_clean), s=100, color='#2E7D32')
axes[0].set_title('Without Outlier — salaries spread nicely across [0,1]')
axes[0].set_xlim(-0.05, 1.05)
axes[0].set_yticks([])

axes[1].scatter(scaled_outlier[:-1], [0]*(len(scaled_outlier)-1), s=100, color='#C62828', label='Normal')
axes[1].scatter([scaled_outlier[-1]], [0], s=140, color='black', marker='X', label='Outlier')
axes[1].set_title('With Outlier — normal salaries crushed near 0')
axes[1].set_xlim(-0.05, 1.05)
axes[1].set_yticks([])
axes[1].legend(loc='upper right')

plt.suptitle('Outlier Sensitivity of MinMaxScaler\nAmol Jagtap | amoljagtap3001@gmail.com', fontsize=11, color='gray')
plt.tight_layout()
plt.savefig('/mnt/user-data/outputs/Day25_outlier_sensitivity.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nConclusion: A single outlier reduced the effective spread of normal')
print('salaries from the full [0,1] range down to a tiny sliver near 0.')
print('The model loses the ability to distinguish between normal salaries.')

---
## Section 7: Normalization vs Standardization — Side-by-Side

**Concept Recap:**
- MinMaxScaler -> bounded range [0,1], sensitive to outliers
- StandardScaler -> mean=0, std=1, unbounded, less sensitive to outliers
- Choice depends on algorithm and data characteristics

In [ ]:
# ============================================================
# ANSWER 7: Normalization vs Standardization Comparison
# ============================================================

minmax = MinMaxScaler()
standard = StandardScaler()

data_minmax   = minmax.fit_transform(data)
data_standard = standard.fit_transform(data)

comparison = pd.DataFrame({
    'Original_Height':   data['Height_cm'],
    'MinMax_Scaled':     data_minmax[:, 0].round(4),
    'Standard_Scaled':   data_standard[:, 0].round(4)
})
print('Side-by-Side Comparison (Height column):')
print(comparison.to_string(index=False))

print('\nMinMaxScaler stats:')
print(f'  min={data_minmax[:,0].min():.4f}, max={data_minmax[:,0].max():.4f}, '
      f'mean={data_minmax[:,0].mean():.4f}, std={data_minmax[:,0].std():.4f}')

print('\nStandardScaler stats:')
print(f'  min={data_standard[:,0].min():.4f}, max={data_standard[:,0].max():.4f}, '
      f'mean={data_standard[:,0].mean():.4f}, std={data_standard[:,0].std():.4f}')

print('\nKey difference:')
print('  MinMaxScaler   -> bounded exactly to [0, 1]')
print('  StandardScaler -> mean exactly 0, std exactly 1, but UNBOUNDED range')

---
## Section 8: Mini End-to-End Pipeline with MinMaxScaler

**Putting it all together!** Build a full Pipeline combining MinMaxScaler with a KNN classifier — the kind of model that absolutely requires feature scaling.

In [ ]:
# ============================================================
# ANSWER 8: Full Scaling + KNN Pipeline
# ============================================================

X = df_pipe[['Age', 'Income']]
y = df_pipe['Buys']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# ----- WITH scaling (Pipeline) -----
pipe_scaled = Pipeline([
    ('scaler', MinMaxScaler()),
    ('knn',    KNeighborsClassifier(n_neighbors=5))
])
pipe_scaled.fit(X_train, y_train)
acc_scaled = pipe_scaled.score(X_test, y_test)

# ----- WITHOUT scaling -----
knn_raw = KNeighborsClassifier(n_neighbors=5)
knn_raw.fit(X_train, y_train)
acc_raw = knn_raw.score(X_test, y_test)

print(f'KNN Accuracy WITHOUT scaling: {acc_raw:.4f}')
print(f'KNN Accuracy WITH MinMax scaling (Pipeline): {acc_scaled:.4f}')

improvement = (acc_scaled - acc_raw) * 100
print(f'\nDifference: {improvement:+.2f} percentage points')

print('\nWhy this happens:')
print('Income has a much larger numeric range than Age. Without scaling,')
print('KNN distance calculations are dominated almost entirely by Income —')
print('Age is effectively ignored. Scaling both features to [0,1] lets')
print('KNN treat Age and Income as equally important neighbors-based features.')

print('\nAmol Jagtap | amoljagtap3001@gmail.com')

---
## Summary & Quick Revision

| Concept | What You Learned |
|---|---|
| Min-Max Formula | `x' = (x - x_min) / (x_max - x_min)` |
| Output range | [0, 1] by default; customizable via `feature_range=(a, b)` |
| Fit on | Training data ONLY — never test data |
| Shape preserved? | Yes — distribution shape stays identical, only range changes |
| Outlier sensitivity | HIGH — one extreme value crushes the rest near 0 or 1 |
| Solution for outliers | Clip outliers first, or use RobustScaler |
| Best for | Neural networks, image pixels, bounded/uniform data |
| sklearn class | `MinMaxScaler()` from `sklearn.preprocessing` |
| Safest workflow | Wrap scaler + model in a `Pipeline` |
| vs StandardScaler | MinMax = bounded [0,1]; Standard = mean 0, std 1, unbounded |

---
> **Notebook by:** Amol Jagtap | amoljagtap3001@gmail.com  
> **Topic:** Day 25 — Feature Scaling: Normalization (MinMaxScaler)